## 03 - Performance

Trial duzeyi performans metrikleri ve katilimci x kosul birimine toplama.

Girdi: NB02'nin `samples_built` / `episodes` ciktilari, NB01'in `trials_clean`'i.
Cikti: `trial_metrics.parquet`, `participant_condition.parquet`.

Bu notebook **istatistiksel test yapmaz**. Friedman / Wilcoxon ve noise
seviyesi karari NB06'nin isi. Buradaki `dz` ve "kac katilimcida ayni yonde"
sayilari betimleyici etki buyuklugu.

Karara baglanan iki acik soru:
- **2** sIQR_theta / sIQR_omega, RMS'in ustune bilgi getiriyor mu
- **3** Episode suresi mi T/T0 mi, sansurlu episode'lar ne olacak

In [1]:
%pip install -q pyyaml pandas numpy pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: C:\Users\elifa\Documents\GitHub\NOROM_Inv_Pendulum\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

# Aktif veri seti. config.yaml -> datasets. Setler birbirine karismaz:
# her set kendi data/<dataset>/{raw,interim,processed} agacinda durur.
DATASET = "pilot2"

ANALYSIS_ROOT = Path.cwd()
for _ in range(4):
    if (ANALYSIS_ROOT / "config.yaml").exists():
        break
    ANALYSIS_ROOT = ANALYSIS_ROOT.parent
sys.path.insert(0, str(ANALYSIS_ROOT))

from src import performance as perf

from src.dataset import load_config, dirs

config, ANALYSIS_ROOT = load_config(DATASET, ANALYSIS_ROOT)
RAW_DIR, INTERIM_DIR, PROCESSED_DIR = dirs(config, ANALYSIS_ROOT)
print(f"veri seti: {config['dataset']}  ({config['dataset_label']})")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

df_samples, episodes, df_trials = perf.load_built(INTERIM_DIR)
print(f"sample  {len(df_samples):,}")
print(f"episode {len(episodes):,}")
print(f"trial   {len(df_trials):,}")
print(f"katilimci {df_samples.participant_id.nunique()}")

veri seti: pilot2  (Pilot 2 -- 9 katilimci, 2-3 Eylul 2026)


sample  623,700
episode 1,332
trial   477
katilimci 9


## 1. Trial duzeyi metrikler

Maske `analysis_include`: active + measurement + qc_pass + focus. Reset
frameleri disarida, her trial tam 1200 sample -- payda sabit.

Dususler **sebebe gore ayriliyor**: `angle` (pole +-60'a vardi) Park'in
Failed'iyla karsilastirilabilir, `track` (cart raydan cikti) Park'ta
karsiligi olmayan ayri bir olay.

In [3]:
trial_df = perf.trial_metrics(df_samples, episodes, config)

print(f"{len(trial_df)} trial x {trial_df.shape[1]} kolon")
print(f"trial basina sample: {trial_df.n_samples.min()} - {trial_df.n_samples.max()}")
print()
display(trial_df.head(3))

450 trial x 28 kolon
trial basina sample: 1200 - 1200



,participant_id,noise_level_id,noise_sigma,trial_id,trial_order,round_index,n_samples,mae_angle_deg,rms_angle_deg,max_abs_angle_deg,siqr_theta_deg,siqr_omega_deg_s,cart_rms_m,control_effort,falls_per_trial,active_s,stab_time_s,stab_pct,n_episodes,n_episodes_censored,mean_episode_s,mean_T_over_T0,n_episodes_done,mean_episode_s_done,mean_T_over_T0_done,mean_theta0_abs_deg,falls_angle_per_trial,falls_track_per_trial
0,P001,N3,0.015,T004,1,1,1200,13.454180,20.370748,63.1305,7.127900,12.62145,0.462728,0.350471,9.0,20.0004,16.983673,84.918365,10,1,2.00005,0.706077,9.0,2.046344,0.714623,3.67685,9.0,0.0
1,P001,N2,0.010,T005,2,1,1200,12.389674,18.180237,61.7793,7.387563,8.42955,0.253240,0.291244,9.0,20.0004,17.817023,89.085115,10,1,2.00004,0.728567,9.0,2.083378,0.749132,4.03758,9.0,0.0
2,P001,N1,0.005,T006,3,1,1200,11.339245,18.410200,61.7561,6.016987,6.53295,0.274306,0.317996,9.0,20.0004,17.650353,88.251765,10,1,2.00003,0.679424,9.0,2.170400,0.736720,3.45247,9.0,0.0


In [4]:
cols = ["mae_angle_deg", "rms_angle_deg", "siqr_theta_deg", "siqr_omega_deg_s",
        "stab_time_s", "falls_per_trial", "falls_angle_per_trial",
        "falls_track_per_trial", "control_effort", "cart_rms_m",
        "n_episodes", "mean_episode_s", "mean_T_over_T0"]
display(trial_df[cols].describe().T.round(3))

,count,mean,std,min,25%,50%,75%,max
mae_angle_deg,450.0,10.684,3.902,2.373,7.628,10.398,13.308,21.704
rms_angle_deg,450.0,14.425,5.331,2.965,10.105,14.304,18.384,27.810
siqr_theta_deg,450.0,7.848,3.138,1.887,5.549,7.416,9.500,20.152
siqr_omega_deg_s,450.0,13.472,7.683,2.915,8.242,10.924,16.692,49.214
stab_time_s,450.0,18.540,1.525,12.850,17.684,18.959,19.850,20.000
falls_per_trial,450.0,1.629,1.878,0.000,0.000,1.000,2.000,9.000
falls_angle_per_trial,450.0,1.407,1.895,0.000,0.000,1.000,2.000,9.000
falls_track_per_trial,450.0,0.222,0.442,0.000,0.000,0.000,0.000,2.000
control_effort,450.0,0.224,0.099,0.062,0.141,0.209,0.298,0.512
cart_rms_m,450.0,1.154,0.556,0.138,0.737,1.082,1.521,3.038


In [5]:
display(perf.check_fall_consistency(trial_df, df_trials))

,trial,sample_toplami_vs_unity,sebep_toplami_vs_unity,unity_toplam,aci,ray
0,450,450,450,733,633.0,100.0


### Kayittaki `within_bounds_time_s` neden kullanilmadi

Unity'nin kolonu failure limitini (60 deg / 5 m) esik aliyor. Trial 20 s ve
neredeyse butun zaman bu limitin icinde geciyor, o yuzden deger her trial'da
tavana yapisik -- koşullari ayirt edemez.

In [6]:
ref = df_trials[df_trials.practice == 0]
print("Unity within_bounds_time_s:")
print(ref.within_bounds_time_s.describe().round(3).to_string())
print()
thr = config["performance"]["stab_angle_deg"]
print(f"Bizim stab_time_s (|theta| <= {thr:.0f} deg):")
print(trial_df.stab_time_s.describe().round(3).to_string())

Unity within_bounds_time_s:
count    450.000
mean      19.973
std        0.031
min       19.850
25%       19.967
50%       19.983
75%       20.000
max       20.000

Bizim stab_time_s (|theta| <= 30 deg):
count    450.000
mean      18.540
std        1.525
min       12.850
25%       17.684
50%       18.959
75%       19.850
max       20.000


## 2. sIQR gereksiz mi

CLAUDE.md'nin gerekcesi: iki katilimcinin maPA'si ayni olabilir ama biri
cogunlukla +-5 derecede durup ara sira +-50'ye giderken digeri surekli
+-15'te olabilir. RMS birinciyi orantisiz cezalandirir.

Test iki asamali: (a) katilimci ici merkezlenmis korelasyon -- katilimcilar
arasi seviye farki korelasyonu sisirdigi icin within surumu kullaniliyor,
(b) sIQR'i RMS uzerine regres edip artigin kosul profili hala oynuyor mu.
Ikisi birden gecerse metrik RMS'in kopyasi.

In [7]:
pc = perf.participant_condition(trial_df)
print(f"{len(pc)} hucre = {pc.participant_id.nunique()} katilimci x "
      f"{pc.noise_level_id.nunique()} kosul, hucre basina {pc.n_trials.unique()} trial")

corr_cols = ["mae_angle_deg", "rms_angle_deg", "siqr_theta_deg",
             "siqr_omega_deg_s", "stab_time_s", "falls_per_trial",
             "control_effort", "cart_rms_m"]
print()
print("Katilimci ici merkezlenmis korelasyon:")
display(perf.metric_correlations(pc, corr_cols))

45 hucre = 9 katilimci x 5 kosul, hucre basina [10] trial

Katilimci ici merkezlenmis korelasyon:


,mae_angle_deg,rms_angle_deg,siqr_theta_deg,siqr_omega_deg_s,stab_time_s,falls_per_trial,control_effort,cart_rms_m
mae_angle_deg,1.000,0.937,0.662,0.496,-0.717,0.086,0.467,0.217
rms_angle_deg,0.937,1.000,0.429,0.394,-0.825,0.249,0.537,0.077
siqr_theta_deg,0.662,0.429,1.000,0.514,-0.204,-0.253,0.264,0.285
siqr_omega_deg_s,0.496,0.394,0.514,1.000,-0.290,0.032,0.646,0.177
stab_time_s,-0.717,-0.825,-0.204,-0.290,1.000,-0.240,-0.406,0.012
falls_per_trial,0.086,0.249,-0.253,0.032,-0.240,1.000,0.488,-0.201
control_effort,0.467,0.537,0.264,0.646,-0.406,0.488,1.000,-0.003
cart_rms_m,0.217,0.077,0.285,0.177,0.012,-0.201,-0.003,1.000


In [8]:
for target in ["siqr_theta_deg", "siqr_omega_deg_s", "mae_angle_deg"]:
    s, prof = perf.redundancy_check(pc, target, "rms_angle_deg", config)
    print(s.to_string())
    display(prof)
    print()

metrik                   siqr_theta_deg
referans                  rms_angle_deg
r_within                          0.429
ham_lineer_kontrast              -0.412
artik_lineer_kontrast            -0.749
korunan_trend                     1.816
trend_esigi                        0.25
gereksiz                          False


,ham_profil,artik_profil
noise_level_id,,
no_noise,0.0340,0.0531
N1,0.0377,0.1344
N2,-0.0547,-0.0739
N3,0.2725,0.2808
N4,-0.2895,-0.3945


metrik                   siqr_omega_deg_s
referans                    rms_angle_deg
r_within                            0.394
ham_lineer_kontrast                 0.252
artik_lineer_kontrast              -0.523
korunan_trend                       2.077
trend_esigi                          0.25
gereksiz                            False


,ham_profil,artik_profil
noise_level_id,,
no_noise,-0.1010,-0.0569
N1,-0.2557,-0.0330
N2,0.2640,0.2200
N3,0.3911,0.4102
N4,-0.2984,-0.5402



metrik                   mae_angle_deg
referans                 rms_angle_deg
r_within                         0.937
ham_lineer_kontrast              0.628
artik_lineer_kontrast           -0.073
korunan_trend                    0.117
trend_esigi                       0.25
gereksiz                          True


,ham_profil,artik_profil
noise_level_id,,
no_noise,-0.0233,0.0166
N1,-0.1978,0.0038
N2,-0.0099,-0.0497
N3,0.0777,0.0950
N4,0.1532,-0.0656


**Karar: sIQR ikisi de NB06'nin metrik setine girmiyor.**

sIQR_theta RMS'in neredeyse kopyasi (r = 0.85) ve noise trendinin sadece
%9'u artikta kaliyor. sIQR_omega ayri bir konstrukt (r = 0.60 -- CLAUDE.md'nin
"acilikten bagimsiz salinim" beklentisi dogru cikti) ama noise trendinin
%79'unu yine RMS acikliyor ve kalan %21 ters isaretli, yani duzensiz.

Ikisi de `trial_metrics.parquet`'te kaliyor: sIQR_omega NB04'te kontrol
mekanizmasini betimlerken ise yarayabilir. Karar metrigi olarak kullanilmiyor.

Kiyas icin maPA da ayni testten geciriliyor -- o da RMS'in kopyasi (r = 0.98),
yani ikisinden sadece biri raporlanmali.

## 3. Sure olcutu: episode suresi mi T/T0 mi

Ludolph'un T/T0'i trial duzeyinde anlamsiz (trial sabit 20 s, T/T0 = 20/T0,
saf theta0 fonksiyonu). Episode duzeyinde anlamli. Burada iki soru birden:

1. Ham episode suresi mi, T0'a bolunmus hali mi -- hangisi baslangic
   acisindan daha bagimsiz
2. Trial sonunda kesilen (sansurlu) episode'lar dahil mi

Sansur cift tarafli sorun: dahil edilirse en iyi denemeler yapay olarak kisa
gorunur, cikarilirsa iyi katilimcinin en iyi episode'lari tamamen silinir.

In [9]:
e = episodes[(episodes.practice == 0) & episodes.qc_pass]
print(f"measurement episode: {len(e)}")
print(f"  sansurlu   : {int(e.censored.sum())} ({100*e.censored.mean():.1f}%)")
print(f"  dususle    : {int(e.ended_in_fall.sum())}")
print()
print("Sansurlu vs sansursuz sure:")
display(e.groupby("censored").duration_s.describe().round(2))

measurement episode: 1183
  sansurlu   : 450 (38.0%)
  dususle    : 733

Sansurlu vs sansursuz sure:


,count,mean,std,min,25%,50%,75%,max
censored,,,,,,,,
False,733.0,6.29,4.6,0.38,2.68,4.82,8.58,19.72
True,450.0,9.75,7.7,0.05,2.75,7.13,20.00,20.00


In [10]:
# Once temel soru: episode suresi bagimsiz bir olcut mu?
# Episode'lar trial'i tam kapliyor (toplam 20 s) ve her dusus bir episode
# sinirî. O halde mean_episode_s = 20 / (dusus + 1) olmali.
pred = trial_df.active_s / trial_df.n_episodes
print("mean_episode_s == active_s / n_episodes :",
      f"max sapma {float((trial_df.mean_episode_s - pred).abs().max()):.2e}")
print("n_episodes == falls_per_trial + 1       :",
      bool((trial_df.n_episodes == trial_df.falls_per_trial + 1).all()))
print()
print("corr(mean_episode_s, 20/(falls+1))  =",
      round(trial_df.mean_episode_s.corr(20 / (trial_df.falls_per_trial + 1)), 4))
print("corr(mean_T_over_T0, mean_episode_s) =",
      round(trial_df.mean_T_over_T0.corr(trial_df.mean_episode_s), 4))

mean_episode_s == active_s / n_episodes : max sapma 5.00e-05
n_episodes == falls_per_trial + 1       : True

corr(mean_episode_s, 20/(falls+1))  = 1.0
corr(mean_T_over_T0, mean_episode_s) = 0.9258


`mean_episode_s` dusus sayisinin **birebir yeniden yazilmis hali**:
korelasyon tam 1.0. Sabit 20 s'lik trial'da episode sayisi = dusus + 1
oldugu icin ortalama episode suresi 20/(dusus+1)'den ibaret. Yeni hicbir
bilgi tasimiyor. `mean_T_over_T0` de onunla 0.93 korelasyonlu.

In [11]:
print("Baslangic acisina duyarlilik (episode duzeyi):")
display(perf.theta0_sensitivity(episodes))
print()
print("Adaylar yan yana (katilimci x kosul duzeyi):")
display(perf.duration_candidate_table(pc, config))

Baslangic acisina duyarlilik (episode duzeyi):


,kume,olcut,n,corr_theta0
0,tum episode,duration_s,1183,-0.032
1,tum episode,duration_over_T0,1183,0.204
2,sansursuz,duration_s,733,-0.020
3,sansursuz,duration_over_T0,733,0.235



Adaylar yan yana (katilimci x kosul duzeyi):


,kosul_acilimi,N4_dz,N4_n_kotu,corr_theta0_pc,eksik_hucre
aday,,,,,
mean_episode_s,1.3461,0.007,3,-0.303,0
mean_episode_s_done,2.8286,-0.796,7,-0.040,0
mean_T_over_T0,0.3865,0.019,3,0.183,0
mean_T_over_T0_done,0.9215,-0.750,7,-0.006,0


**Karar: bagimsiz bir sure metrigi kullanilmiyor; `falls_per_trial` yeterli
istatistik. T/T0 reddedildi.**

Uc aday da eleniyor, ayri ayri sebeplerle:

- **`mean_episode_s`** dusus sayisinin deterministik donusumu (yukarida),
  ayri bir metrik degil.
- **`mean_T_over_T0`** T0'a bolmek duzeltmiyor, **fazla duzeltiyor**: episode
  duzeyinde ham surenin theta0 korelasyonu -0.075 iken bolunmus hali +0.141,
  katilimci x kosul duzeyinde ise 0.229'a karsi **0.645**. Yani Ludolph'un
  normalizasyonu bizim tasarimimizda kirliligi azaltmiyor, artiriyor.
- **Sansursuz surumler** hayatta kalma yanliligi tasiyor. Sansurlu episode
  demek "trial sonuna kadar dusmedi" demek, yani en iyi denemeler. Onlari
  atinca no_noise'un ortalamasi 12.10 s'den 7.38 s'ye duserek en **dusuk**
  kosul haline geliyor -- sacma bir sonuc. N4 etkisi de bu yuzden isaret
  degistiriyor (dz -1.03 -> +0.25).

Ludolph'un sure temelli olcutunu duzgun kullanmak icin sag sansuru ele alan
bir survival analizi (Kaplan-Meier / Cox) gerekir; 1756 episode'un 600'u
sansurlu, bu goz ardi edilecek bir oran degil. NB03'un kapsami disinda,
gerekirse NB05'te yapilir. Pilot karari icin `falls_per_trial` ayni bilgiyi
tasiyor ve yorumu net.

## 4. Katilimci x kosul

Analiz birimi. Her hucre 10 measurement trial'in ortalamasi.

`baseline_farki` = kosul - no_noise, katilimci basina eslesmis fark.
`dz` = mean(fark) / sd(fark). `n_kotu` = 12 katilimcinin kacinda fark
metrigin kotu yonunde. Hepsi betimleyici, p degeri NB06'da.

In [12]:
labels = perf.condition_labels(trial_df)
print(" | ".join(labels.values()))
print()
display(perf.condition_table(pc))

no_noise (σ=0.000) | N1 (σ=0.005) | N2 (σ=0.010) | N3 (σ=0.015) | N4 (σ=0.020)



noise_level_id,yon,no_noise,N1,N2,N3,N4
mae_angle_deg,-1,10.661 ±0.968,10.486 ±1.097,10.674 ±0.936,10.762 ±1.223,10.837 ±1.012
rms_angle_deg,-1,14.365 ±1.336,14.119 ±1.57,14.486 ±1.292,14.399 ±1.644,14.757 ±1.444
siqr_theta_deg,-1,7.882 ±0.757,7.886 ±0.781,7.793 ±0.69,8.121 ±0.985,7.559 ±0.626
siqr_omega_deg_s,-1,13.371 ±1.954,13.216 ±1.932,13.736 ±2.135,13.863 ±2.482,13.173 ±1.97
stab_time_s,1,18.544 ±0.355,18.55 ±0.412,18.604 ±0.387,18.543 ±0.415,18.459 ±0.398
falls_per_trial,-1,1.689 ±0.564,1.6 ±0.642,1.678 ±0.479,1.544 ±0.567,1.633 ±0.504
falls_angle_per_trial,-1,1.456 ±0.58,1.367 ±0.655,1.478 ±0.494,1.3 ±0.547,1.433 ±0.503
falls_track_per_trial,-1,0.233 ±0.082,0.233 ±0.047,0.2 ±0.053,0.244 ±0.056,0.2 ±0.06
control_effort,0,0.225 ±0.031,0.224 ±0.031,0.225 ±0.028,0.221 ±0.028,0.223 ±0.029
cart_rms_m,0,1.203 ±0.123,1.076 ±0.083,1.16 ±0.106,1.182 ±0.096,1.15 ±0.106


In [13]:
for m in ["mae_angle_deg", "stab_time_s", "falls_per_trial",
          "falls_angle_per_trial", "falls_track_per_trial", "control_effort"]:
    print(f"--- {m}  ({perf.METRIC_INFO[m][0]}) ---")
    display(perf.baseline_agreement(pc, m, config))

--- mae_angle_deg  (Mean |theta| (deg)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.174,-0.203,4,9
N2,0.013,0.014,4,9
N3,0.101,0.089,3,9
N4,0.176,0.184,5,9


--- stab_time_s  (Stabilizasyon suresi (s / 20 s)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,0.006,0.011,3,9
N2,0.060,0.125,3,9
N3,-0.001,-0.003,3,9
N4,-0.086,-0.176,5,9


--- falls_per_trial  (Dusus / trial) ---


...

,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.089,-0.203,3,9
N2,0.022,0.052,5,9
N3,-0.156,-0.567,2,9
N4,-0.022,-0.080,5,9


--- falls_track_per_trial  (Ray kaynakli dusus / trial) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,0.000,0.000,4,9
N2,-0.033,-0.153,2,9
N3,0.011,0.036,4,9
N4,-0.033,-0.167,3,9


--- control_effort  (Control effort (RMS u)) ---


,baseline_farki,dz,n_kotu,n
kosul,,,,
N1,-0.001,-0.020,NaN,9
N2,0.001,0.029,NaN,9
N3,-0.004,-0.163,NaN,9
N4,-0.002,-0.109,NaN,9


### Baslangic acisi kirliligi

Randomizasyon sorunu (CLAUDE.md 1): butun katilimcilar ayni sabit RNG
dizisinden okuyor. Kosul ortalamalarinda kalan dengesizlik burada.

In [14]:
display(pc.groupby("noise_level_id", observed=True)
          .mean_theta0_abs_deg.agg(["mean", "std"]).round(3))

,mean,std
noise_level_id,,
no_noise,3.592,0.333
N1,3.501,0.346
N2,3.599,0.641
N3,3.927,0.553
N4,3.677,0.496


## Cikti

In [15]:
info = perf.save_outputs(trial_df, pc, INTERIM_DIR)
for k, v in info.items():
    print(f"{k:34} ({v:,} satir)")

trial_metrics.parquet              (450 satir)
participant_condition.parquet      (45 satir)


## Ozet

**Metrik seti (NB06'ya giden).** Yonu net ve birbirinin kopyasi olmayanlar:

| Metrik | Yon | Not |
|---|---|---|
| `mae_angle_deg` | dusuk iyi | RMS ile r = 0.98; ikisinden biri secilmeli, maPA daha okunakli |
| `stab_time_s` | yuksek iyi | kendi esigimiz (30 deg); Unity'nin `within_bounds_time_s`'i tavana yapisik |
| `falls_angle_per_trial` | dusuk iyi | Park'in Failed'iyla karsilastirilabilir olan |
| `control_effort` | belirsiz | tie-breaker, tek basina "iyi/kotu" demiyor |
| `cart_rms_m` | belirsiz | tie-breaker |

**Disarida birakilanlar:** `siqr_theta_deg` ve `siqr_omega_deg_s` (noise
trendini RMS zaten acikliyor -- 2), `mean_episode_s` / `mean_T_over_T0` ve
sansursuz surumleri (dusus sayisinin donusumu ya da theta0 kirli -- 3),
`falls_track_per_trial` (kosulla ilgisiz gorunuyor, dz'ler +-0.12 icinde ve
Park karsilastirmasindan zaten cikariliyor). Hepsi `trial_metrics.parquet`'te
duruyor, sadece karar setinde degil.

**Betimleyici tablo ne diyor.** Tum ana metriklerde ayni sekil: no_noise ile
N1 birbirine yapisik, N2'den itibaren monoton bozulma. maPA'da N2/N3/N4
katilimcilarin 11/12'sinde baseline'dan kotu (dz 0.96-1.18). N1'de fark yok
(dz -0.03, 6/12). Yani **orta seviyede iyilesme, yani U sekli yok** --
stochastic resonance beklentisinin tersi. Testler NB06'da.

**Uyari.** `mean_theta0_abs_deg` kosullar arasi 3.53-3.85 deg araliginda,
en zor baslangiclar no_noise'da. Yanlilik bulgunun aleyhine calisiyor, yani
gozlenen bozulmayi sisirmis olamaz.